# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nirvik-49/Week-1-FlyRank-AI-Assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Binary Classification (with optional Scoring for ranking priority)

Why: The goal is to separate web pages into two actionable buckets: 1 (Decaying / Needs Priority Content Refresh) and 0 (Stable or Growing / No Immediate Action). While ranking or scoring can quantify the severity of traffic loss, the core operational decision for an SEO team is binary: Do we flag this page for a rewrite this sprint, or leave it alone?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

print("Task Type: Binary Classification")
print("Target Class 1: Decaying (Needs Refresh)")
print("Target Class 0: Stable / Growing (No Action)")

Task Type: Binary Classification
Target Class 1: Decaying (Needs Refresh)
Target Class 0: Stable / Growing (No Action)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I would predict: is_decaying (Binary target: 1 or 0).

Label Origin: Derived Proxy Label (Observed Outcome based on temporal rules).
Since raw organic search logs rarely contain explicit "decayed" labels, the target is constructed by comparing recent performance against historical baselines:

* Proxy Definition: A page is labeled is_decaying = 1 if its organic clicks over the last 30 days drop by ≥ 20% compared to the same 30-day period in the previous quarter, accompanied by a decline in average search position (position getting higher/worse). Otherwise, is_decaying = 0.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Proxy Target Logic Simulation
# Assuming df has 'clicks_last_30d', 'clicks_prev_30d', and 'avg_position_delta'

def generate_proxy_label(df):
    pct_change = (df['clicks_last_30d'] - df['clicks_prev_30d']) / (df['clicks_prev_30d'] + 1e-5)
    # Decaying if clicks dropped by 20% or more AND rank deteriorated
    df['is_decaying'] = np.where((pct_change <= -0.20) & (df['avg_position_delta'] > 0), 1, 0)
    return df

print("Proxy label logic defined successfully.")

Proxy label logic defined successfully.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Defensible Metric: Precision@K (where K = top 20 prioritized pages per update cycle) & Recall on Decaying Class

* Primary Metric (Precision@K / Precision for Class 1): ≥ 85%

* Why: Content teams have limited engineering/editorial bandwidth. If the model flags 20 pages for a rewrite, at least 17 of them must actually be suffering from true structural organic decay (not seasonal noise). False positives waste expensive writer time.

* Secondary Metric (Recall for Class 1): ≥ 75%. We want to catch the majority of decaying high-value assets before traffic drops to zero.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.metrics import precision_score, recall_score

# Example verification metric function
y_true = np.array([1, 1, 0, 1, 0, 0, 1, 0, 1, 1])
y_pred = np.array([1, 1, 0, 1, 0, 0, 1, 0, 0, 1])

prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)

print(f"Target Precision (Class 1): {prec:.2f} (Goal: >= 0.85)")
print(f"Target Recall (Class 1):    {rec:.2f} (Goal: >= 0.75)")

Target Precision (Class 1): 1.00 (Goal: >= 0.85)
Target Recall (Class 1):    0.83 (Goal: >= 0.75)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique Page URL aggregated over a rolling 30-day observation window.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Load starter data slice or construct the exact Unit of Analysis DataFrame schema
# Replace 'data.csv' with your starter dataset path if available in repo
data = {
    'url': [
        'https://example.com/blog/flask-guide',
        'https://example.com/blog/sql-optimization',
        'https://example.com/pricing',
        'https://example.com/blog/python-basics'
    ],
    'clicks_last_30d': [1200, 450, 3100, 890],
    'clicks_prev_30d': [1850, 460, 3150, 1200],
    'impressions_last_30d': [25000, 10000, 50000, 15000],
    'avg_position_last_30d': [14.2, 4.1, 2.3, 11.8],
    'avg_position_delta': [3.5, -0.2, 0.1, 2.1], # Positive means rank worsened
    'ctr': [0.048, 0.045, 0.062, 0.059]
}

df_unit_of_analysis = pd.DataFrame(data)

# Compute target column for demonstration
df_unit_of_analysis['pct_change'] = (df_unit_of_analysis['clicks_last_30d'] - df_unit_of_analysis['clicks_prev_30d']) / df_unit_of_analysis['clicks_prev_30d']
df_unit_of_analysis['is_decaying'] = np.where((df_unit_of_analysis['pct_change'] <= -0.20) & (df_unit_of_analysis['avg_position_delta'] > 0), 1, 0)

print("Unit of Analysis DataFrame (One row = One URL):")
df_unit_of_analysis[['url', 'clicks_last_30d', 'pct_change', 'avg_position_delta', 'is_decaying']]

Unit of Analysis DataFrame (One row = One URL):


,url,clicks_last_30d,pct_change,avg_position_delta,is_decaying
0,https://example.com/blog/flask-guide,1200,-0.351351,3.5,1
1,https://example.com/blog/sql-optimization,450,-0.021739,-0.2,0
2,https://example.com/pricing,3100,-0.015873,0.1,0
3,https://example.com/blog/python-basics,890,-0.258333,2.1,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed if-statement:

1. Seasonality vs. Decay: A simple rule like if clicks drop > 20% then decay fails during expected seasonal dips (e.g., lower traffic in December). An ML model learns non-linear interactions across seasonality, impressional trends, and SERP position shifts simultaneously.

2. Algorithmic SERP Shifts: Search engine algorithm updates frequently shift Search Engine Results Page (SERP) features (such as AI Overviews or Featured Snippets). A page might lose clicks while holding rank 1 because a snippet took space. ML models separate CTR degradation from structural keyword decay across complex feature combinations.

3. Threshold Rigidity: Fixed rules break on multi-intent queries. A 10% drop on a top-converting enterprise URL may be far more critical than a 40% drop on a low-value legacy post. ML classifiers learn non-linear decision boundaries that static if/else statements cannot capture without endless manual tuning.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Demonstrating complexity: Non-linear relationship between CTR, Position, and Decay
# Fixed rules miss subtle non-linear boundaries
print("Fixed Rule Limitation Example:")
print("- Simple Rule: clicks_drop > 20%")
print("- Missed False Positive: Seasonal traffic drops where position remains unchanged.")
print("- Missed False Negative: Gradual decay where position slips by 0.5 every week, staying under hard thresholds.")

Fixed Rule Limitation Example:
- Simple Rule: clicks_drop > 20%
- Missed False Positive: Seasonal traffic drops where position remains unchanged.
- Missed False Negative: Gradual decay where position slips by 0.5 every week, staying under hard thresholds.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.